## Configuration

In [0]:
LANDING_BASE = "s3://trading-signals-landing"

OHLCV_LANDING = f"{LANDING_BASE}/ohlcv"
MACRO_LANDING = f"{LANDING_BASE}/macro"

CHECKPOINT_BASE = f"{LANDING_BASE}/_checkpoints"
OHLCV_CHECKPOINT = f"{CHECKPOINT_BASE}/ohlcv"
MACRO_CHECKPOINT = f"{CHECKPOINT_BASE}/macro"

# Target tables
OHLCV_TABLE = "raghavan_trading_signals.bronze.raw_ohlcv"
MACRO_TABLE = "raghavan_trading_signals.bronze.raw_macro"

## Bronze OHLCV - Autoloader Ingestion - Write OHLCV Stream to Bronze Delta Table

In [0]:
from pyspark.sql.functions import current_timestamp, col

ohlcv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{OHLCV_CHECKPOINT}/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(OHLCV_LANDING)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
(
    ohlcv_stream.writeStream
    .format("delta")                                 # Write as Delta table
    .outputMode("append")                            # Only append new rows (never overwrite)
    .option("checkpointLocation", OHLCV_CHECKPOINT)  # Track progress (exactly-once guarantee)
    .option("mergeSchema", "true")                   # Allow schema evolution on the write side too
    .trigger(availableNow=True)                      # Process all available files, then stop
    .toTable(OHLCV_TABLE)                            # Write to Unity Catalog table
)

## Bronze Macro - Autoloader Ingestion - Write Macro Stream to Bronze Delta Table

In [0]:
macro_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation",
            f"{MACRO_CHECKPOINT}/schema")
    .option("cloudFiles.schemaEvolutionMode",
            "addNewColumns")
    .load(MACRO_LANDING)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
(
    macro_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", MACRO_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(MACRO_TABLE)
)

## Verify the Bronze Tables

In [0]:
print("=== Bronze OHLCV ===")
ohlcv_df = spark.table(OHLCV_TABLE)
print(f"Row count: {ohlcv_df.count():,}")
print(f"Columns: {ohlcv_df.columns}")
display(ohlcv_df.limit(5))

In [0]:
print("=== Bronze Macro ===")
macro_df = spark.table(MACRO_TABLE)
print(f"Row count: {macro_df.count():,}")
print(f"Columns: {macro_df.columns}")
display(macro_df.limit(5))

In [0]:
display(spark.sql(f"DESCRIBE EXTENDED {OHLCV_TABLE}"))

In [0]:
display(spark.sql(f"DESCRIBE EXTENDED {MACRO_TABLE}"))